# Animation Studio \u2014 Character LoRA Training (Phase 1c)

Trains a character-consistency LoRA with kohya sd-scripts `flux_train_network.py` on a Colab GPU runtime, then benchmarks it against the studio identity scorer and syncs the artifacts back to the AnimationStudio GitHub repo.

## What this notebook does (run top-to-bottom, edit only the **Settings** cell)

1. Settings (runtime parameters + token placeholders)
2. GPU check \u2014 fails fast with guidance on no-GPU / under-powered runtimes
3. Clones the AnimationStudio repo, installs the studio, pins kohya `sd-scripts` to a fixed upstream commit (supply-chain pin)
4. Downloads the four FLUX.1-dev model files (gated \u2192 requires `HF_TOKEN`)
5. Builds the training dataset from curated character assets (`scripts/train_lora.py build-dataset`)
6. Trains the LoRA (`accelerate launch flux_train_network.py`, VRAM profiles for A100 / T4 / 12 GB)
7. Generates sample images with a diffusers Flux pipeline and benchmarks them with `LoRABenchmark` + `IdentityScorerProvider(light=False)`
8. Registers the version in the LoRA registry and promotes it when the benchmark gate passes
9. Syncs `*.safetensors` + benchmark report + registry to GitHub (or downloads them manually)

## Prerequisites

- **GPU Colab runtime**: T4 GPU (free tier) or A100 GPU. CPU-only runtimes fail fast in cell 2.
- **Hugging Face token** (`hf_...`) for the gated **FLUX.1-dev** model \u2014 accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev, then paste the token in the Settings cell.
- **GitHub fine-grained PAT** with *Contents: Read and write* on the AnimationStudio repo (only needed when `SYNC_TO_GITHUB = True`, cell 9).

## Security note

This notebook commits with **empty token placeholders**. `HF_TOKEN` and `GITHUB_TOKEN` are runtime session values only \u2014 they are never written back into the notebook and never committed to git.

## Time expectations

Hours-scale on free T4 (\u22485 s/it @512 px with the fp8 profile). Watch Colab session limits; a failed GPU session means re-running from the top (artifacts already synced are skipped idempotently).

In [ ]:
#@title 1. Settings

import os
import pathlib
import subprocess
import sys

# --- Repo -------------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}
BRANCH = "master"  #@param ["master", "colab-gpu"]

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
SD_SCRIPTS_DIR = f"{WORK}/sd-scripts"

# --- Training run -----------------------------------------------------------
CHARACTER_ID = "lily-bunny"  #@param {type:"string"}
CHARACTER_TITLE = "Lily Bunny"  # Universe directory display name (Universe/Characters/<title>)
VRAM_PROFILE = "t4-16g"  #@param ["a100-24g", "t4-16g", "t4-12gb-swap16"]

# Training artifacts live inside the repo clone so the sync cell can commit them.
TRAINING_ROOT = f"{REPO}/training"
OUTPUT_DIR = f"{REPO}/Universe/Characters/{CHARACTER_TITLE}/lora"
SAMPLES_DIR = f"{WORK}/training-samples"
MODEL_DIR = f"{WORK}/models"
REGISTRY_PATH = f"{TRAINING_ROOT}/lora_registry.json"

# --- GitHub sync ------------------------------------------------------------
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}

# --- Secrets (runtime-only; never commit real values) -----------------------
GITHUB_TOKEN = ""  #@param {type:"string"}
HF_TOKEN = ""      #@param {type:"string"}

In [ ]:
#@title 2. GPU check (fail fast)

import shutil

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU detected. This notebook trains FLUX.1-dev LoRAs and needs a "
        "GPU Colab runtime: Runtime > Change runtime type > T4 GPU (free) or A100 GPU, "
        "then re-run from cell 1."
    )
subprocess.run(["nvidia-smi"], check=True)  # device summary

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "torch reports CUDA unavailable. Verify the runtime accelerator is set to a "
        "GPU (Runtime > Change runtime type), then re-run from cell 1."
    )

device = torch.cuda.get_device_name(0)
COMPUTE = torch.cuda.get_device_capability()
COMPUTE_MAJOR = COMPUTE[0]
print(
    f"Device: {device} | CUDA {torch.version.cuda} | torch {torch.__version__} "
    f"| compute capability {COMPUTE[0]}.{COMPUTE[1]}"
)
if COMPUTE_MAJOR < 8:
    print("NOTE: Turing/older GPUs have no native bf16; fp16 raises NaN on Flux.")
    print("The training cell forces the fp8_base path (emulated, ~5 s/it @512 px) \u2014 expect hours-scale runs.")

In [ ]:
#@title 3. Clone the repo, install the studio, pin kohya sd-scripts

def run(argv):
    print("+ " + " ".join(str(a) for a in argv))
    subprocess.run([str(a) for a in argv], check=True)

# --- AnimationStudio --------------------------------------------------------
if not os.path.isdir(REPO):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
run(["git", "-C", REPO, "checkout", BRANCH])
run(["git", "-C", REPO, "pull", "origin", BRANCH])
os.chdir(REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])

# --- kohya sd-scripts (pinned to a fixed upstream commit for reproducibility) ----
SD_SCRIPTS_URL = "https://github.com/kohya-ss/sd-scripts.git"
SD_SCRIPTS_PIN = "37a1cbbc5725ed2a3575506e7bd2001c9908ac92"  # main @ 2026-07-23
if not os.path.isdir(SD_SCRIPTS_DIR):
    run(["git", "clone", SD_SCRIPTS_URL, SD_SCRIPTS_DIR])
run(["git", "-C", SD_SCRIPTS_DIR, "checkout", SD_SCRIPTS_PIN])
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{SD_SCRIPTS_DIR}/requirements.txt"])

In [ ]:
#@title 4. Download the four FLUX.1-dev model files

# Four files required by flux_train_network.py (see sd-scripts FLUX LoRA guide):
#   flux1-dev.safetensors + ae.safetensors  <- black-forest-labs/FLUX.1-dev (gated)
#   clip_l.safetensors + t5xxl_fp16.safetensors <- comfyanonymous/flux_text_encoders
pathlib.Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

from huggingface_hub import hf_hub_download

def _download(repo_id, filename):
    try:
        return hf_hub_download(
            repo_id, filename, local_dir=MODEL_DIR, token=HF_TOKEN or None
        )
    except Exception as exc:
        raise RuntimeError(
            f"Failed to download {filename} from {repo_id}. Gated FLUX.1-dev files "
            f"require an HF token that accepted the model license "
            f"(https://huggingface.co/settings/tokens + "
            f"https://huggingface.co/black-forest-labs/FLUX.1-dev). Cause: {exc}"
        ) from exc

MODEL_FLUX = _download("black-forest-labs/FLUX.1-dev", "flux1-dev.safetensors")
MODEL_AE = _download("black-forest-labs/FLUX.1-dev", "ae.safetensors")
MODEL_CLIP_L = _download("comfyanonymous/flux_text_encoders", "clip_l.safetensors")
MODEL_T5XXL = _download("comfyanonymous/flux_text_encoders", "t5xxl_fp16.safetensors")

for p in (MODEL_FLUX, MODEL_AE, MODEL_CLIP_L, MODEL_T5XXL):
    size_mb = os.path.getsize(p) / 1024 / 1024
    print(f"{os.path.basename(p)}: {size_mb:.0f} MiB")
    if os.path.getsize(p) == 0:
        raise RuntimeError(f"Downloaded file is empty: {p}")

In [ ]:
#@title 5. Build the training dataset from curated assets

# build-dataset pulls identity-locked curated assets (states: approved,production)
# from catalog.db and writes dataset_config.toml, train/ + val/ subsets and
# baselines/ reference images under TRAINING_ROOT.
# Requires at least 20 curated images per character; below that the CLI exits
# non-zero printing per-state counts \u2014 promote more assets in the Review UI first.
try:
    run([
        sys.executable, "scripts/train_lora.py", "build-dataset",
        "--db", "catalog.db",
        "--character-id", CHARACTER_ID,
        "--output-root", TRAINING_ROOT,
        "--min-images", "20",
        "--max-images", "40",
    ])
except subprocess.CalledProcessError as exc:
    print(
        "\nDataset build failed (see output above). If the curated set is under the "
        "20-image minimum, promote additional assets to approved/production via the "
        "Review UI, then re-run this cell."
    )
    raise

DATASET_CONFIG = f"{TRAINING_ROOT}/dataset_config.toml"
print("\nDataset config:", DATASET_CONFIG)

In [ ]:
#@title 6. Train the LoRA (accelerate + flux_train_network.py)

# Next recommended version for this character (v0.1 on a fresh registry).
from src.training_engine.version_store import load_registry

registry = load_registry(pathlib.Path(REGISTRY_PATH))
next_version = registry.recommend_next(CHARACTER_ID, "minor")
version_str = str(next_version)
print(f"Training {CHARACTER_ID} \u2192 {version_str} (registry-recommended next version)")

# VRAM profiles (cited in 01c-RESEARCH.md):
#   a100-24g       : batch 2, native bf16, no fp8, no swap
#   t4-16g         : batch 1 + --fp8_base + --blocks_to_swap 8  (Turing bf16 emulated)
#   t4-12gb-swap16 : batch 1 + --fp8_base + --blocks_to_swap 16 + AdamW8bit
VRAM_PROFILES = {
    "a100-24g":       {"batch_size": 2, "fp8_base": False, "blocks_to_swap": 0},
    "t4-16g":         {"batch_size": 1, "fp8_base": True,  "blocks_to_swap": 8},
    "t4-12gb-swap16": {"batch_size": 1, "fp8_base": True,  "blocks_to_swap": 16},
}
if VRAM_PROFILE not in VRAM_PROFILES:
    raise ValueError(f"Unknown VRAM_PROFILE: {VRAM_PROFILE!r} (choose from {list(VRAM_PROFILES)})")
profile = VRAM_PROFILES[VRAM_PROFILE]

# Pitfall 6: ever force fp8_base on compute < 8 even for the a100 profile.
use_fp8 = profile["fp8_base"] or COMPUTE_MAJOR < 8
if use_fp8 and not profile["fp8_base"]:
    print("NOTE: compute capability < 8 \u2192 forcing --fp8_base despite selected profile.")

LORA_NAME = f"{CHARACTER_ID}_{version_str}"
output_name = LORA_NAME  # -> Universe/Characters/Lily Bunny/lora/lily-bunny_v0.1.safetensors

cmd = [
    "accelerate", "launch", "--num_cpu_threads_per_process", "1",
    f"{SD_SCRIPTS_DIR}/flux_train_network.py",
    "--pretrained_model_name_or_path", MODEL_FLUX,
    "--clip_l", MODEL_CLIP_L,
    "--t5xxl", MODEL_T5XXL,
    "--ae", MODEL_AE,
    "--dataset_config", DATASET_CONFIG,
    "--output_dir", OUTPUT_DIR,
    "--output_name", output_name,
    "--save_model_as", "safetensors",
    "--network_module", "networks.lora_flux",
    "--network_dim", "32",
    "--network_alpha", "32",
    "--learning_rate", "1e-4",
    "--optimizer_type", "AdamW8bit",
    "--lr_scheduler", "constant",
    "--max_train_epochs", "10",
    "--mixed_precision", "bf16",
    "--save_precision", "bf16",
    "--seed", "42",
    "--gradient_checkpointing",
    "--sdpa",
    "--train_batch_size", str(profile["batch_size"]),
    "--cache_latents",
    "--cache_latents_to_disk",
    "--cache_text_encoder_outputs",
    "--cache_text_encoder_outputs_to_disk",
    "--guidance_scale", "1.0",
    "--timestep_sampling", "flux_shift",
    "--model_prediction_type", "raw",
    "--max_data_loader_n_workers", "2",
]
if use_fp8:
    cmd += ["--fp8_base"]
if profile["blocks_to_swap"] > 0:
    cmd += ["--blocks_to_swap", str(profile["blocks_to_swap"])]

# Long-running cell: hours-scale on free T4. Do not interrupt; watch session limits.
run(cmd)

LORA_PATH = pathlib.Path(OUTPUT_DIR) / f"{LORA_NAME}.safetensors"
if not LORA_PATH.exists():
    raise RuntimeError(f"Training finished but {LORA_PATH} is missing \u2014 check the accelerate log above.")
print(f"\nSaved: {LORA_PATH}")

In [ ]:
#@title 7. Generate samples and benchmark the LoRA

# Sample generation: diffusers Flux pipeline with the trained LoRA loaded.
# Prompts mirror BenchmarkConfig.test_prompts (single source of truth).
from diffusers import FluxPipeline
from src.training_engine.benchmark import BenchmarkConfig

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN or None,
)
pipe.load_lora_weights(OUTPUT_DIR, weight_name=f"{LORA_NAME}.safetensors")
pipe.to("cuda")

pathlib.Path(SAMPLES_DIR).mkdir(parents=True, exist_ok=True)
sample_images = []
for i, (prompt, _asset_type) in enumerate(BenchmarkConfig().test_prompts):
    gen = torch.Generator("cuda").manual_seed(42 + i)
    image = pipe(
        prompt=f"{CHARACTER_TITLE}, {prompt}",
        num_inference_steps=28,
        guidance_scale=3.5,
        generator=gen,
    ).images[0]
    out_path = pathlib.Path(SAMPLES_DIR) / f"sample-{i + 1:02d}.png"
    image.save(out_path)
    sample_images.append(out_path)
    print(f"sample-{i + 1:02d}.png <- {prompt!r}")

# Benchmark against the identity scorer baseline (full plugin stack on Colab).
from src.training_engine.benchmark import LoRABenchmark
from src.training_engine.scorer_adapter import IdentityScorerProvider

benchmark = LoRABenchmark(
    scorer_provider=IdentityScorerProvider(light=False),  # full DINOv2/CLIP plugins
    config=BenchmarkConfig(baseline_dir=pathlib.Path(f"{TRAINING_ROOT}/baselines")),
)
result = benchmark.evaluate(
    lora_path=LORA_PATH,
    character_id=CHARACTER_ID,
    test_images=sample_images,
)
report = benchmark.report(result)
print(report)
print(f"Gate: composite >= 0.90 AND weight coverage 100% -> passed={result.passed}")

REPORT_PATH = pathlib.Path(OUTPUT_DIR) / f"benchmark_report_{version_str}.md"
REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Report saved: {REPORT_PATH}")

In [ ]:
#@title 8. Register and promote the version

# Real training completion on Colab \u2014 dry-run registration semantics do NOT apply.
# Register always; promote only when the benchmark gate passed (composite >= 0.90
# with full weight coverage, per CHAR-07 / Phase 1c success criteria).
from src.training_engine.versioning import LoRAVersion

training_config_used = {
    "base_model": "black-forest-labs/FLUX.1-dev",
    "network_module": "networks.lora_flux",
    "network_dim": 32,
    "network_alpha": 32,
    "learning_rate": 1e-4,
    "optimizer_type": "AdamW8bit",
    "lr_scheduler": "constant",
    "max_train_epochs": 10,
    "mixed_precision": "bf16",
    "resolution": 1024,  # must match cache in dataset_config.toml
    "seed": 42,
    "vram_profile": VRAM_PROFILE,
    "dataset_config": DATASET_CONFIG,
    "output_dir": OUTPUT_DIR,
}

benchmark_scores = {dim.name: dim.score for dim in result.dimensions}
benchmark_scores["composite"] = result.composite_score
benchmark_scores["passed"] = result.passed

registry = load_registry(pathlib.Path(REGISTRY_PATH))
registry.register(
    character_id=CHARACTER_ID,
    version=next_version,
    file_path=str(LORA_PATH),
    training_config=training_config_used,
    benchmark_scores=benchmark_scores,
)

if result.passed:
    registry.promote(CHARACTER_ID, next_version)
    print(f"\u2705 Promoted {CHARACTER_ID} {version_str} to production (benchmark gate passed).")
else:
    print(
        f"\u274c Benchmark gate FAILED for {version_str} (composite={result.composite_score:.2%}). "
        f"Version is registered but NOT promoted \u2014 retrain with more/better data."
    )

print("Registry:", REGISTRY_PATH)

In [ ]:
#@title 9. Sync artifacts to GitHub (or manual download)

from datetime import datetime

if SYNC_TO_GITHUB:
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "SYNC_TO_GITHUB is on but GITHUB_TOKEN is empty. Create a fine-grained PAT "
            "with Contents: Read and write on AnimationStudio, paste it into Settings, "
            "or set SYNC_TO_GITHUB = False for manual download."
        )
    sys.path.insert(0, f"{REPO}/colab")
    from git_sync import _basic_auth_header

    run(["git", "config", "user.name", GIT_NAME])
    run(["git", "config", "user.email", GIT_EMAIL])

    # Extended add vs Phase 4 pattern: LoRA safetensors + benchmark report
    # (both live in Universe/Characters/Lily Bunny/lora/) + registry + dataset manifest.
    run(["git", "add", f"Universe/Characters/{CHARACTER_TITLE}/lora"])
    for rel in ["training/lora_registry.json", "training/dataset_config.toml", "training/metadata.json"]:
        if pathlib.Path(rel).exists():
            run(["git", "add", rel])

    staged = subprocess.run(
        ["git", "diff", "--cached", "--name-only"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    if not staged:
        print("Nothing new to commit \u2014 artifacts already synced on a prior run (idempotent).")
    else:
        stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        run(["git", "commit", "-m", f"feat(training): {CHARACTER_ID} {version_str} LoRA + benchmark ({stamp})"])
        run(["git", "-c", f"http.extraheader={_basic_auth_header(GITHUB_TOKEN)}", "push", "origin", BRANCH])
        print("Pushed to GitHub.")
else:
    from google.colab import files

    files.download(str(LORA_PATH))
    files.download(str(REPORT_PATH))
    print("Manual download started for LoRA + benchmark report.")

## Next steps

- **Verify the commit**: check GitHub for `Universe/Characters/Lily Bunny/lora/` (safetensors + benchmark report) and `training/lora_registry.json`.
- **Review remaining assets**: promote more curated assets through the Review UI (`nursery review` / HTTP UI) if the curated set was under 20 images \u2014 then re-run from cell 5.
- **Use the LoRA downstream**: load `lily-bunny_v1.0.safetensors` in the Phase 4 generation pipeline (ComfyUI / diffusers `load_lora_weights`) with the identity-locked prompt system.
- **Other characters**: set `CHARACTER_ID` + `CHARACTER_TITLE` in cell 1 (e.g. `penny-pig` / `Penny Pig`), ensure curated assets exist in the catalog, and re-run.
- **Benchmark evidence**: this notebook\u2019s benchmark report + registry JSON are the recorded evidence for the deferred-human verification of the LOv1 production LoRA criterion (CHAR-07).
- **Session limits**: on free T4 expect hours-scale; if the session dies mid-training, re-run from the top \u2014 dataset rebuild is cheap and sync is idempotent.